In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objs as go
import numpy as np

In [2]:
profiles = pd.read_excel("..\AY2627_S1_CA2 datasets & meta data\CA2 datasets & meta data\Client profiles.xlsx")
profiles.head()

,CLIENT ID,TYPE,SG-BASED,OVERSEAS,COUNTRY OF OVERSEAS CLIENT,COMMENCEMENT DATE,STAFF STRENGTH,SECTOR
0,G1-001,Govt,Yes,NaN,,2020-01-19 00:00:00,More than 200,Finance
1,G1-002,Govt,Yes,NaN,,2020-05-19 00:00:00,More than 200,Security
2,G1-003,Govt,Yes,NaN,,2021-07-18 00:00:00,> 200,Transportation
3,G1-005,Govt,Y,NaN,NaN,2022-02-11 00:00:00,> 200,Security
4,G1-006,Govt,Y,NaN,NaN,2022-03-18 00:00:00,50 ~ 200,Legal


Things to clean:\
    - columns using different values to represent the same info(Sg-based,staff strength)\
    - getting rid of missing values (Country of overseas client)

In [3]:
profiles[['COUNTRY OF OVERSEAS CLIENT','OVERSEAS','SG-BASED']].value_counts()

COUNTRY OF OVERSEAS CLIENT  OVERSEAS  SG-BASED
Malaysia                    Yes                   6
                            Y                     3
India                       Yes                   1
China                       Y                     1
Indonesia                   Yes                   1
Malaysian                   Y                     1
Name: count, dtype: int64

In [4]:
# ---- Clean Client Profiles ----
# Strip stray leading/trailing whitespace on text columns
obj_cols = ['SG-BASED', 'OVERSEAS', 'COUNTRY OF OVERSEAS CLIENT',
            'STAFF STRENGTH', 'SECTOR', 'TYPE', 'CLIENT ID']
for c in obj_cols:
    profiles[c] = profiles[c].apply(lambda v: v.strip() if isinstance(v, str) else v)

# SG-BASED / OVERSEAS: standardise to explicit Y / N ('Yes' and 'Y' -> 'Y', blanks -> 'N')
def yes_no(v):
    return 'Y' if str(v).strip().upper() in ('Y', 'YES') else 'N'
profiles['SG-BASED'] = profiles['SG-BASED'].apply(yes_no)
profiles['OVERSEAS'] = profiles['OVERSEAS'].apply(yes_no)

# COUNTRY OF OVERSEAS CLIENT: unify blanks to NaN, fix 'Malaysian' -> 'Malaysia'
profiles['COUNTRY OF OVERSEAS CLIENT'] = (
    profiles['COUNTRY OF OVERSEAS CLIENT']
    .replace(r'^\s*$', np.nan, regex=True)       # empty / whitespace-only -> NaN
    .replace('Malaysian', 'Malaysia')
)

# Derived COUNTRY: domestic clients -> 'Singapore', otherwise the overseas country
profiles['COUNTRY'] = profiles['COUNTRY OF OVERSEAS CLIENT'].fillna('Singapore')

# STAFF STRENGTH: normalise synonym labels, then derive an ORG SIZE bucket.
# Metadata rule: 1~49 = Small, 50~200 = Medium, >200 = Large.
# 'Less than 200' is ambiguous (spans Small & Medium) -> kept as 'Unspecified'.
profiles['STAFF STRENGTH'] = profiles['STAFF STRENGTH'].replace(
    {'More than 200': '> 200', 'Less than 49': '1 ~ 49'})
size_map = {'1 ~ 49': 'Small', '50 ~ 200': 'Medium',
            '> 200': 'Large', 'Less than 200': 'Unspecified'}
profiles['ORG SIZE'] = profiles['STAFF STRENGTH'].map(size_map)
assert profiles['ORG SIZE'].notna().all(), \
    profiles.loc[profiles['ORG SIZE'].isna(), 'STAFF STRENGTH'].unique()

# COMMENCEMENT DATE: mix of datetime & string -> uniform datetime64
profiles['COMMENCEMENT DATE'] = pd.to_datetime(profiles['COMMENCEMENT DATE'], errors='coerce')
assert profiles['COMMENCEMENT DATE'].notna().all(), "Unparseable COMMENCEMENT DATE remains"

profiles.head()

,CLIENT ID,TYPE,SG-BASED,OVERSEAS,COUNTRY OF OVERSEAS CLIENT,COMMENCEMENT DATE,STAFF STRENGTH,SECTOR,COUNTRY,ORG SIZE
0,G1-001,Govt,Y,N,NaN,2020-01-19,> 200,Finance,Singapore,Large
1,G1-002,Govt,Y,N,NaN,2020-05-19,> 200,Security,Singapore,Large
2,G1-003,Govt,Y,N,NaN,2021-07-18,> 200,Transportation,Singapore,Large
3,G1-005,Govt,Y,N,NaN,2022-02-11,> 200,Security,Singapore,Large
4,G1-006,Govt,Y,N,NaN,2022-03-18,50 ~ 200,Legal,Singapore,Medium


In [5]:
profiles['STAFF STRENGTH'].value_counts()

STAFF STRENGTH
> 200            62
50 ~ 200         57
1 ~ 49           22
Less than 200     7
Name: count, dtype: int64

In [6]:
profiles['SG-BASED'].value_counts()

SG-BASED
Y    114
N     34
Name: count, dtype: int64

In [7]:
profiles.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 148 entries, 0 to 147
Data columns (total 10 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   CLIENT ID                   148 non-null    object        
 1   TYPE                        148 non-null    object        
 2   SG-BASED                    148 non-null    object        
 3   OVERSEAS                    148 non-null    object        
 4   COUNTRY OF OVERSEAS CLIENT  34 non-null     object        
 5   COMMENCEMENT DATE           148 non-null    datetime64[ns]
 6   STAFF STRENGTH              148 non-null    object        
 7   SECTOR                      148 non-null    object        
 8   COUNTRY                     148 non-null    object        
 9   ORG SIZE                    148 non-null    object        
dtypes: datetime64[ns](1), object(9)
memory usage: 11.7+ KB


In [8]:
survey = pd.read_excel("..\AY2627_S1_CA2 datasets & meta data\CA2 datasets & meta data\Client survey.xlsx")
survey.head()

,YEAR,CLIENT ID,PRESALES AND PARTNERSHIP,TECHNICAL EXPERTISE,PROJECT DELIVERY,POST-SALES SUPPORT,NPS RATING
0,2021,G1-001,3,4,4,5,8
1,2021,G1-002,3,4,4,2,7
2,2021,G1-003,5,4,3,5,8
3,2021,G1-001,3,4,4,5,8
4,2021,G1-002,3,4,4,2,7


In [9]:
survey['CLIENT ID'].value_counts()

CLIENT ID
G1-001    9
G1-002    9
G1-003    9
G1-006    7
N1-001    5
         ..
P3-027    1
P3-028    1
P3-029    1
P3-030    1
P3-031    1
Name: count, Length: 143, dtype: int64

In [10]:
survey.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 432 entries, 0 to 431
Data columns (total 7 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   YEAR                      432 non-null    int64 
 1   CLIENT ID                 432 non-null    object
 2   PRESALES AND PARTNERSHIP  432 non-null    int64 
 3   TECHNICAL EXPERTISE       432 non-null    int64 
 4   PROJECT DELIVERY          432 non-null    int64 
 5   POST-SALES SUPPORT        432 non-null    int64 
 6   NPS RATING                432 non-null    int64 
dtypes: int64(6), object(1)
memory usage: 23.8+ KB


In [11]:
survey.isnull().sum()

YEAR                        0
CLIENT ID                   0
PRESALES AND PARTNERSHIP    0
TECHNICAL EXPERTISE         0
PROJECT DELIVERY            0
POST-SALES SUPPORT          0
NPS RATING                  0
dtype: int64

In [12]:
# ---- Clean Client Survey ----
# Drop 30 exact duplicate rows, then drop orphan CLIENT IDs not in Client Profiles
survey = survey.drop_duplicates()
survey = survey[survey['CLIENT ID'].isin(profiles['CLIENT ID'])].reset_index(drop=True)

# Derived NPS category per respondent (Meta Data thresholds)
def nps_category(score):
    if score >= 9:
        return 'Promoter'    # 9-10
    elif score >= 7:
        return 'Passive'     # 7-8
    return 'Detractor'       # 6 or less
survey['NPS CATEGORY'] = survey['NPS RATING'].apply(nps_category)

survey.head()

,YEAR,CLIENT ID,PRESALES AND PARTNERSHIP,TECHNICAL EXPERTISE,PROJECT DELIVERY,POST-SALES SUPPORT,NPS RATING,NPS CATEGORY
0,2021,G1-001,3,4,4,5,8,Passive
1,2021,G1-002,3,4,4,2,7,Passive
2,2021,G1-003,5,4,3,5,8,Passive
3,2021,N1-001,4,4,4,3,8,Passive
4,2021,N1-002,5,3,4,3,7,Passive


In [13]:
RC = pd.read_excel("..\AY2627_S1_CA2 datasets & meta data\CA2 datasets & meta data\Revenue and COGS.xlsx")
RC.head()

,CLIENT ID,YEAR,REVENUE,HARDWARE,SOFTWARE,MANPOWER
0,G1-001,2021,148180,17321,25562,41201
1,G1-001,2022,197313,23604,32875,50085
2,G1-001,2023,349899,51332,71695,89089
3,G1-001,2024,268821,40969,56452,76545
4,G1-001,2025,138115,14516,20889,28968


In [14]:
# ---- Clean Revenue and COGS ----
# Drop 21 exact duplicate rows, then drop orphan CLIENT IDs not in Client Profiles
# (P1-010, P1-018, P1-035 -> metadata says such records can be ignored)
RC = RC.drop_duplicates()
RC = RC[RC['CLIENT ID'].isin(profiles['CLIENT ID'])].reset_index(drop=True)

# Derived cost / profitability metrics (formulas from Meta Data)
RC['COGS'] = RC['HARDWARE'] + RC['SOFTWARE'] + RC['MANPOWER']
RC['GROSS PROFIT'] = RC['REVENUE'] - RC['COGS']
RC['GROSS MARGIN %'] = (RC['GROSS PROFIT'] / RC['REVENUE'] * 100).round(2)
RC['GROSS MARKUP %'] = (RC['GROSS PROFIT'] / RC['COGS'] * 100).round(2)
RC['LOSS'] = RC['GROSS PROFIT'] < 0   # negative gross profit (all NPO clients) -> flagged, kept

RC.head()

,CLIENT ID,YEAR,REVENUE,HARDWARE,SOFTWARE,MANPOWER,COGS,GROSS PROFIT,GROSS MARGIN %,GROSS MARKUP %,LOSS
0,G1-001,2021,148180,17321,25562,41201,84084,64096,43.26,76.23,False
1,G1-001,2022,197313,23604,32875,50085,106564,90749,45.99,85.16,False
2,G1-001,2023,349899,51332,71695,89089,212116,137783,39.38,64.96,False
3,G1-001,2024,268821,40969,56452,76545,173966,94855,35.29,54.53,False
4,G1-001,2025,138115,14516,20889,28968,64373,73742,53.39,114.55,False


In [15]:
RC.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 392 entries, 0 to 391
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   CLIENT ID       392 non-null    object 
 1   YEAR            392 non-null    int64  
 2   REVENUE         392 non-null    int64  
 3   HARDWARE        392 non-null    int64  
 4   SOFTWARE        392 non-null    int64  
 5   MANPOWER        392 non-null    int64  
 6   COGS            392 non-null    int64  
 7   GROSS PROFIT    392 non-null    int64  
 8   GROSS MARGIN %  392 non-null    float64
 9   GROSS MARKUP %  392 non-null    float64
 10  LOSS            392 non-null    bool   
dtypes: bool(1), float64(2), int64(7), object(1)
memory usage: 31.1+ KB


In [16]:
RC.isnull().sum()

CLIENT ID         0
YEAR              0
REVENUE           0
HARDWARE          0
SOFTWARE          0
MANPOWER          0
COGS              0
GROSS PROFIT      0
GROSS MARGIN %    0
GROSS MARKUP %    0
LOSS              0
dtype: int64

In [17]:
profiles.head()

,CLIENT ID,TYPE,SG-BASED,OVERSEAS,COUNTRY OF OVERSEAS CLIENT,COMMENCEMENT DATE,STAFF STRENGTH,SECTOR,COUNTRY,ORG SIZE
0,G1-001,Govt,Y,N,NaN,2020-01-19,> 200,Finance,Singapore,Large
1,G1-002,Govt,Y,N,NaN,2020-05-19,> 200,Security,Singapore,Large
2,G1-003,Govt,Y,N,NaN,2021-07-18,> 200,Transportation,Singapore,Large
3,G1-005,Govt,Y,N,NaN,2022-02-11,> 200,Security,Singapore,Large
4,G1-006,Govt,Y,N,NaN,2022-03-18,50 ~ 200,Legal,Singapore,Medium


In [18]:
survey.head()

,YEAR,CLIENT ID,PRESALES AND PARTNERSHIP,TECHNICAL EXPERTISE,PROJECT DELIVERY,POST-SALES SUPPORT,NPS RATING,NPS CATEGORY
0,2021,G1-001,3,4,4,5,8,Passive
1,2021,G1-002,3,4,4,2,7,Passive
2,2021,G1-003,5,4,3,5,8,Passive
3,2021,N1-001,4,4,4,3,8,Passive
4,2021,N1-002,5,3,4,3,7,Passive


In [19]:
RC.head()

,CLIENT ID,YEAR,REVENUE,HARDWARE,SOFTWARE,MANPOWER,COGS,GROSS PROFIT,GROSS MARGIN %,GROSS MARKUP %,LOSS
0,G1-001,2021,148180,17321,25562,41201,84084,64096,43.26,76.23,False
1,G1-001,2022,197313,23604,32875,50085,106564,90749,45.99,85.16,False
2,G1-001,2023,349899,51332,71695,89089,212116,137783,39.38,64.96,False
3,G1-001,2024,268821,40969,56452,76545,173966,94855,35.29,54.53,False
4,G1-001,2025,138115,14516,20889,28968,64373,73742,53.39,114.55,False


In [20]:
# ---- Export cleaned datasets ----
import os
os.makedirs('cleaned', exist_ok=True)
profiles.to_csv('cleaned/client_profiles_clean.csv', index=False)
RC.to_csv('cleaned/revenue_cogs_clean.csv', index=False)
survey.to_csv('cleaned/client_survey_clean.csv', index=False)
print('Exported cleaned files:', os.listdir('cleaned'))

Exported cleaned files: ['client_profiles_clean.csv', 'client_survey_clean.csv', 'revenue_cogs_clean.csv']


In [21]:
# ---- Validation checks on cleaned data ----
# Client Profiles
assert len(profiles) == 148
assert set(profiles['SG-BASED'].unique()) <= {'Y', 'N'}
assert set(profiles['OVERSEAS'].unique()) <= {'Y', 'N'}
assert ((profiles['SG-BASED'] == 'Y') ^ (profiles['OVERSEAS'] == 'Y')).all(), "SG/OVERSEAS not mutually exclusive"
assert set(profiles['ORG SIZE'].unique()) <= {'Small', 'Medium', 'Large', 'Unspecified'}
assert profiles['ORG SIZE'].notna().all()
assert pd.api.types.is_datetime64_any_dtype(profiles['COMMENCEMENT DATE'])
assert profiles['COMMENCEMENT DATE'].notna().all()
assert profiles['COUNTRY'].notna().all()

# Revenue and COGS
assert RC.duplicated().sum() == 0
assert RC['CLIENT ID'].isin(profiles['CLIENT ID']).all()
assert (RC['COGS'] == RC['HARDWARE'] + RC['SOFTWARE'] + RC['MANPOWER']).all()
assert np.isfinite(RC['GROSS MARGIN %']).all() and np.isfinite(RC['GROSS MARKUP %']).all()
assert int(RC['LOSS'].sum()) == 5

# Client Survey
rating_cols = ['PRESALES AND PARTNERSHIP', 'TECHNICAL EXPERTISE', 'PROJECT DELIVERY', 'POST-SALES SUPPORT']
assert survey.duplicated().sum() == 0
assert survey['CLIENT ID'].isin(profiles['CLIENT ID']).all()
assert survey[rating_cols].isin([1, 2, 3, 4, 5]).all().all()
assert survey['NPS RATING'].between(0, 10).all()
assert set(survey['NPS CATEGORY'].unique()) <= {'Promoter', 'Passive', 'Detractor'}

print('All validation checks passed.')
print(f"profiles: {profiles.shape} | RC: {RC.shape} | survey: {survey.shape}")
print(f"LOSS rows (negative gross profit): {int(RC['LOSS'].sum())}")
print('\nORG SIZE:\n', profiles['ORG SIZE'].value_counts())
print('\nNPS CATEGORY:\n', survey['NPS CATEGORY'].value_counts())

All validation checks passed.
profiles: (148, 10) | RC: (392, 11) | survey: (392, 8)
LOSS rows (negative gross profit): 5

ORG SIZE:
 ORG SIZE
Large          62
Medium         57
Small          22
Unspecified     7
Name: count, dtype: int64

NPS CATEGORY:
 NPS CATEGORY
Passive      181
Promoter     165
Detractor     46
Name: count, dtype: int64


## DB 1 (Elliot)

#### Graph 1

#### Graph 2

#### Graph 3

## DB 2 (Zafran)

#### Graph 1

In [22]:
fig = px.sunburst(profiles,
    path=["COUNTRY","TYPE","SECTOR",],
    # values='STAFF STRENGTH',
    title='sunburst plot showing the country, type and sector',
    height = 700,
    width = 700
)
fig.show()

#### Graph 2

In [23]:
merged_df1 = pd.merge(profiles, RC, on='CLIENT ID', how='inner') #used AI for this in Scatterplot of year against country mean profit chat
ProfitPerCountry = merged_df1.groupby(['COUNTRY', 'YEAR'])['GROSS PROFIT'].sum().reset_index()

fig = go.Figure()

for country in ProfitPerCountry['COUNTRY'].unique():
    subset = ProfitPerCountry[ProfitPerCountry['COUNTRY'] == country]
    fig.add_trace(
        go.Scatter(
            x=subset['YEAR'],
            y=subset['GROSS PROFIT'],
            name=country,
            mode='lines+markers'
        )
    )

fig.update_layout(
    title='Mean Profit Per Country Timeline',
    xaxis_title='Year',
    yaxis_title='Mean Profit',
    height=1000
)

fig.show()

#### Graph 3

## DB 3 (name)

#### Graph 1

#### Graph 2

#### Graph 3